# CLIF APS Connect

# Setup

* Load libraries, create required directories, and load the config file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
import sys
import clifpy
import yaml
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"clifpy version: {clifpy.__version__}")
print(f"clifpy location: {clifpy.__file__}")
warnings.filterwarnings('ignore')

In [ ]:
# Load configuration
config_path = "../config/config.json"
with open(config_path, 'r') as f:
    config = json.load(f)

# Load config
with open('../config/aps_clif_data_requirements.yaml', 'r') as f:
    aps_config = yaml.safe_load(f)

## import outlier json
# with open('../config/outlier_config.json', 'r', encoding='utf-8') as f:
#     outlier_cfg = json.load(f)

print(f"\n=� Configuration:")
print(f"   Data directory: {config['tables_path']}")
print(f"   File type: {config['file_type']}")
print(f"   Timezone: {config['timezone']}")

import os
# Create output directories if they do not exist
os.makedirs("../output/final/graphs", exist_ok=True)
os.makedirs("../output/intermediate", exist_ok=True)
OUTPUT_DIR = "./output/final"
FINAL_OUTPUT_DIR = "../output/final"
INTERMEDIATE_OUTPUT_DIR = "../output/intermediate"

# CLIF Clinical Trials 

In [ ]:
trials_fp = Path(config['tables_path']) / f"clif_clinical_trials.{config['file_type']}"
trials_df = pd.read_parquet(trials_fp)
# Convert patient_id to object (string type), lowercase trial_name, and convert enrollment_dttm to datetime
trials_df['patient_id'] = trials_df['patient_id'].astype(str)
trials_df['trial_name'] = trials_df['trial_name'].str.lower()
trials_df['enrollment_dttm'] = pd.to_datetime(trials_df['enrollment_dttm'])
# Keep only rows where trial_name contains 'aps'
# trials_df = trials_df[trials_df['trial_name'].str.contains('aps')]

# Get all patient_ids in a list
trial_patient_ids = trials_df['patient_id'].tolist()

In [ ]:
trials_df

# CLIF Core

* Patient, Hospitalization

In [ ]:
print("\n" + "=" * 80)
print("Loading CLIF Tables")
print("=" * 80)

from clifpy.clif_orchestrator import ClifOrchestrator
# Initialize ClifOrchestrator
clif = ClifOrchestrator(
    data_directory=config['tables_path'],
    filetype=config['file_type'],
    timezone=config['timezone'],
    output_directory = OUTPUT_DIR
)

clif.load_table(
        'patient',
        filters={'patient_id': list(trial_patient_ids)}
    )

clif.load_table(
        'hospitalization',
        filters={'patient_id': list(trial_patient_ids)}
    )

trial_hosp_ids = clif.hospitalization.df['hospitalization_id'].tolist()

In [ ]:
patient_required_columns = aps_config['patient_required_columns']
hosp_required_columns = aps_config['hospitalization_required_columns']

patient_df = clif.patient.df.loc[:, patient_required_columns]
hosp_df = clif.hospitalization.df.loc[:, hosp_required_columns]

# Merge patient and hospitalization tables on 'patient_id'
static_df = hosp_df.merge(
    patient_df,
    on='patient_id',
    how='left'
)

In [ ]:
# SANITY CHECK
patient_df_ids = patient_df['patient_id'].astype(str)
patient_filtered = patient_df[patient_df_ids.isin(trial_patient_ids)]
print(f"APS patients to find: {len(trial_patient_ids)}")
print(f"APS patients found in patient table: {len(patient_filtered)}")

# Show which APS patients were found/not found
found_ids = set(patient_filtered['patient_id'].astype(str))
requested_ids = set(trial_patient_ids)
missing_ids = requested_ids - found_ids

if missing_ids:
    print(f"Missing APS patient IDs: {sorted(missing_ids)}")
else:
    print("All APS patients found in patient table")

QUESTION-- Should we only keep hospitalization ids closest to the enrollment_dttm for that patient?

In [ ]:
# cleanup
print(clif.get_tables_obj_list())
# Delete the patient and hospitalization table objects
clif.patient = None
clif.hospitalization = None
del patient_df, hosp_df
# Verify they're gone
print(clif.get_tables_obj_list())

In [ ]:
# Merge static_df with trials_df on 'patient_id'
static_df = static_df.merge(
    trials_df,
    on='patient_id',
    how='left'
)

# Wide Dataset

In [ ]:
# Define columns for each table
columns_config = {
    'labs': aps_config['labs_required_columns'],
    'vitals': aps_config['vitals_required_columns'],
    'medication_admin_continuous': aps_config['meds_continuous_required_columns'],
    'medication_admin_intermittent': aps_config['meds_intermittent_required_columns']
}

# Define filters for each table (hospitalization_id + category filters)
filters_config = {
    'labs': {
        'hospitalization_id': list(trial_hosp_ids),
        'lab_category': aps_config['labs_of_interest']
    },
    'vitals': {
        'hospitalization_id': list(trial_hosp_ids),
        'vital_category': aps_config['vitals_of_interest']
    },
    'medication_admin_continuous': {
        'hospitalization_id': list(trial_hosp_ids),
        'med_category': aps_config['meds_continuous_of_interest']
    },
    'medication_admin_intermittent': {
        'hospitalization_id': list(trial_hosp_ids),
        'med_category': aps_config['meds_intermittent_of_interest']
    }
}

# Load all tables in one call
clif.initialize(
    tables=['labs', 'vitals', 'medication_admin_continuous', 
            'medication_admin_intermittent'],
    columns=columns_config,
    filters=filters_config
)

In [ ]:
# =============================================================================
# Extract DataFrames from loaded tables
# =============================================================================
labs_df = clif.labs.df.copy()
vitals_df = clif.vitals.df.copy()
meds_cont_df = clif.medication_admin_continuous.df.copy()
meds_int_df = clif.medication_admin_intermittent.df.copy()

# =============================================================================
# Pivot Vitals: narrow → wide
# =============================================================================
vitals_wide = vitals_df.pivot_table(
    index=['hospitalization_id', 'recorded_dttm'],
    columns='vital_category',
    values='vital_value',
    aggfunc='first'
).reset_index()

# Rename datetime column to event_dttm and flatten column names
vitals_wide = vitals_wide.rename(columns={'recorded_dttm': 'event_dttm'})
vitals_wide.columns = ['hospitalization_id', 'event_dttm'] + \
    [f'vital_{col}' for col in vitals_wide.columns[2:]]

# =============================================================================
# Pivot Labs: narrow → wide
# =============================================================================
labs_wide = labs_df.pivot_table(
    index=['hospitalization_id', 'lab_result_dttm'],
    columns='lab_category',
    values='lab_value_numeric',
    aggfunc='first'
).reset_index()

# Rename datetime column to event_dttm
labs_wide = labs_wide.rename(columns={'lab_result_dttm': 'event_dttm'})
labs_wide.columns = ['hospitalization_id', 'event_dttm'] + \
    [f'lab_{col}' for col in labs_wide.columns[2:]]

# =============================================================================
# Pivot Meds Continuous: narrow → wide
# =============================================================================
meds_cont_wide = meds_cont_df.pivot_table(
    index=['hospitalization_id', 'admin_dttm'],
    columns='med_category',
    values='med_dose',
    aggfunc='first'
).reset_index()

# Rename datetime column to event_dttm
meds_cont_wide = meds_cont_wide.rename(columns={'admin_dttm': 'event_dttm'})
meds_cont_wide.columns = ['hospitalization_id', 'event_dttm'] + \
    [f'med_cont_{col}' for col in meds_cont_wide.columns[2:]]

# =============================================================================
# Pivot Meds Intermittent: narrow → wide
# =============================================================================
meds_int_wide = meds_int_df.pivot_table(
    index=['hospitalization_id', 'admin_dttm'],
    columns='med_category',
    values='med_dose',
    aggfunc='first'
).reset_index()

# Rename datetime column to event_dttm
meds_int_wide = meds_int_wide.rename(columns={'admin_dttm': 'event_dttm'})
meds_int_wide.columns = ['hospitalization_id', 'event_dttm'] + \
    [f'med_int_{col}' for col in meds_int_wide.columns[2:]]

# =============================================================================
# Merge all wide tables on hospitalization_id + event_dttm
# =============================================================================
wide_df = vitals_wide.merge(
    labs_wide,
    on=['hospitalization_id', 'event_dttm'],
    how='outer'
)

wide_df = wide_df.merge(
    meds_cont_wide,
    on=['hospitalization_id', 'event_dttm'],
    how='outer'
)

wide_df = wide_df.merge(
    meds_int_wide,
    on=['hospitalization_id', 'event_dttm'],
    how='outer'
)

# Sort by hospitalization and time
wide_df = wide_df.sort_values(['hospitalization_id', 'event_dttm']).reset_index(drop=True)

print(f"Final wide dataframe: {wide_df.shape}")
print(f"Columns: {list(wide_df.columns)}")

# Respiratory Support

In [ ]:
clif.load_table(
        'respiratory_support',
        columns= aps_config['respiratory_support_required_columns'], 
        filters={'hospitalization_id': list(trial_hosp_ids)}
    )

In [ ]:
clif.respiratory_support = clif.respiratory_support.waterfall()

In [ ]:
# =============================================================================
# Respiratory Support (already wide format)
# =============================================================================
resp_df = clif.respiratory_support.df.copy()

# Rename datetime column to event_dttm
resp_df = resp_df.rename(columns={'recorded_dttm': 'event_dttm'})

# Add prefix to all columns except hospitalization_id and event_dttm
resp_cols_to_rename = [col for col in resp_df.columns if col not in ['hospitalization_id', 'event_dttm']]
resp_df = resp_df.rename(columns={col: f'resp_{col}' for col in resp_cols_to_rename})

# Merge with wide_df
wide_df = wide_df.merge(
    resp_df,
    on=['hospitalization_id', 'event_dttm'],
    how='outer'
)

# Re-sort by hospitalization and time
wide_df = wide_df.sort_values(['hospitalization_id', 'event_dttm']).reset_index(drop=True)

print(f"Final wide dataframe with resp support: {wide_df.shape}")
print(f"Unique patients: {wide_df['hospitalization_id'].map(static_df.set_index('hospitalization_id')['patient_id']).nunique()}")
print(f"Unique hospitalizations: {wide_df['hospitalization_id'].nunique()}")

# ADT

In [ ]:
clif.load_table(
        'adt',
        columns= aps_config['adt_required_columns'], 
        filters={'hospitalization_id': list(trial_hosp_ids)}
    )
adt_df = clif.adt.df

In [ ]:
# =============================================================================
# Add ADT rows to wide_df (location transitions as events)
# =============================================================================
adt_df['in_dttm'] = pd.to_datetime(adt_df['in_dttm'])
adt_df['out_dttm'] = pd.to_datetime(adt_df['out_dttm'])

# Sort ADT by hospitalization and in_dttm
adt_sorted = adt_df.sort_values(['hospitalization_id', 'in_dttm'])

# Create ADT wide format with in_dttm as event_dttm
adt_wide = adt_df[['hospitalization_id', 'in_dttm', 'location_category', 'location_name']].copy()
adt_wide = adt_wide.rename(columns={'in_dttm': 'event_dttm'})

# Add prefix to ADT columns
adt_wide = adt_wide.rename(columns={
    'location_category': 'adt_location_category',
    'location_name': 'adt_location_name'
})

# Merge with wide_df
wide_df = wide_df.merge(
    adt_wide,
    on=['hospitalization_id', 'event_dttm'],
    how='outer'
)

# Sort by hospitalization and time
wide_df = wide_df.sort_values(['hospitalization_id', 'event_dttm']).reset_index(drop=True)

# Forward fill location within each hospitalization so every row knows the current location
wide_df['adt_location_category'] = wide_df.groupby('hospitalization_id')['adt_location_category'].ffill()
wide_df['adt_location_name'] = wide_df.groupby('hospitalization_id')['adt_location_name'].ffill()

print(f"wide_df shape: {wide_df.shape}")

# Microbiology

In [ ]:
clif.load_table(
        'microbiology_culture',
        columns= aps_config['microbiology_culture_required_columns'], 
        filters={'hospitalization_id': list(trial_hosp_ids)}
    )
micro_culture_df = clif.microbiology_culture.df

In [ ]:
clif.load_table(
        'microbiology_nonculture',
        columns= aps_config['microbiology_nonculture_required_columns'], 
        filters={'hospitalization_id': list(trial_hosp_ids)}
    )
micro_nonculture_df = clif.microbiology_nonculture.df

# What day is it?

Join the wide_df with trials_df to get enrollment_dttm. 
If the enrollment date is the same as event date, then it is day 0. 

In [ ]:
# =============================================================================
# Add patient_id and enrollment_dttm to wide_df, calculate relative day
# =============================================================================

# Get patient_id for each hospitalization from static_df
hosp_patient_map = static_df[['hospitalization_id', 'patient_id', 'record_id']].drop_duplicates()

# Add patient_id to wide_df
wide_df = wide_df.merge(
    hosp_patient_map,
    on='hospitalization_id',
    how='left'
)

In [ ]:
# Merge enrollment_dttm directly from trials_df at patient level
wide_df = wide_df.merge(
    trials_df[['patient_id',  'enrollment_dttm']],
    on='patient_id',
    how='left'
)

In [ ]:
# Ensure enrollment_dttm is datetime and localize to same timezone as event_dttm
wide_df['enrollment_dttm'] = pd.to_datetime(wide_df['enrollment_dttm'])

# Get timezone from event_dttm and apply to enrollment_dttm
event_tz = wide_df['event_dttm'].dt.tz
if event_tz is not None:
    wide_df['enrollment_dttm'] = wide_df['enrollment_dttm'].dt.tz_localize(event_tz)

# Extract just the date portion from event_dttm for comparison
wide_df['event_date'] = wide_df['event_dttm'].dt.normalize()

# Calculate day relative to enrollment (Day 0 = enrollment date)
wide_df['day'] = (wide_df['event_date'] - wide_df['enrollment_dttm']).dt.days

# Clean up temporary column
wide_df = wide_df.drop(columns=['event_date'])

# Reorder columns to put patient_id and day near the front
cols = ['record_id', 'patient_id', 'hospitalization_id', 'event_dttm', 'enrollment_dttm', 'day'] + \
       [c for c in wide_df.columns if c not in ['record_id','patient_id', 'hospitalization_id', 'event_dttm', 'enrollment_dttm', 'day']]
wide_df = wide_df[cols]
# sanity check
print(f"\nDay range: {wide_df['day'].min()} to {wide_df['day'].max()}")

In [ ]:
for p in wide_df.columns:
    print(p)

# Save intermediate data

In [ ]:
# Save wide_df and static_df as parquet files in INTERMEDIATE_DIR
wide_df.to_parquet(f"{INTERMEDIATE_OUTPUT_DIR}/wide_df.parquet", index=False)
static_df.to_parquet(f"{INTERMEDIATE_OUTPUT_DIR}/static_df.parquet", index=False)
micro_culture_df.to_parquet(f"{INTERMEDIATE_OUTPUT_DIR}/micro_culture_df.parquet", index=False)
micro_nonculture_df.to_parquet(f"{INTERMEDIATE_OUTPUT_DIR}/micro_nonculture_df.parquet", index=False)